# SEO Audit Tool

Crawls a website and lists SEO problems under **Technical**, **On-Page**, **Off-Page** and **Spelling & Grammar**, with a fix for each.

**How to use:** run the cells from top to bottom (click the ▶ button on each, or use *Runtime → Run all*).
1. Step 1 installs the tool (takes about 30 seconds).
2. Step 2: type your website address and press ▶. It prints scores and shows a download button for the Excel report.

## Step 1 - Install

In [ ]:
#@title Install the tool { display-mode: "form" }
#@markdown Leave the token empty if the GitHub repository is public. For a private repository, paste a GitHub personal access token (with "repo" read access).
GITHUB_TOKEN = ""  #@param {type:"string"}

import os, subprocess, sys
REPO = "yogeshkhute123/Assignment-No-"
BRANCH = "claude/gsc-report-automation-t5igp4"   # change to "main" once the pull request is merged
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
if not os.path.isdir("/content/seo_tool"):
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{auth}github.com/{REPO}.git", "/content/seo_tool"],
                       capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("Could not download the tool. If the repository is private, paste a GitHub token above and run again.\n" + r.stderr[-300:])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/seo_tool/requirements.txt"], check=True)
os.chdir("/content/seo_tool")
sys.path.insert(0, "/content/seo_tool")
print("Installed. Now go to Step 2.")

## Step 2 - Run the audit

In [ ]:
#@title Audit a website { display-mode: "form" }
WEBSITE = "https://example.com"  #@param {type:"string"}
MAX_PAGES = 100  #@param {type:"slider", min:10, max:500, step:10}
CHECK_OUTBOUND_LINKS = True  #@param {type:"boolean"}
#@markdown Words the spell checker should accept (brand names, jargon), separated by commas:
EXTRA_WORDS = ""  #@param {type:"string"}
#@markdown Tick to upload a backlink export (CSV from Ahrefs / Semrush / Moz / Search Console) for the off-page check:
UPLOAD_BACKLINKS_CSV = False  #@param {type:"boolean"}

import os
from seo_audit import run_audit, report

backlinks = None
if UPLOAD_BACKLINKS_CSV:
    from google.colab import files
    up = files.upload()
    backlinks = next(iter(up), None)

custom = None
if EXTRA_WORDS.strip():
    custom = "/content/extra_words.txt"
    with open(custom, "w") as fh:
        fh.write("\n".join(w.strip() for w in EXTRA_WORDS.split(",") if w.strip()))

site, issues, scores = run_audit(WEBSITE, max_pages=MAX_PAGES, check_external=CHECK_OUTBOUND_LINKS,
                                 backlink_csv=backlinks, custom_dict=custom, verbose=False)
out = "/content/seo_report"
report.write_xlsx(out + ".xlsx", site, issues, scores)
report.write_html(out + ".html", site, issues, scores)

print(f"Pages crawled: {len(site.pages)}" + ("  (limit reached - raise MAX_PAGES for more)" if site.truncated else ""))
print("Scores (0-100):", ", ".join(f"{k}: {v}" for k, v in scores.items()))
print()
for row in report.group(issues)[:25]:
    print(f"[{row['category'].split(' (')[0]:<9}] {row['severity']:<6} {row['title']}  ({len(row['urls']) or 1})")
if len(report.group(issues)) > 25:
    print(f"... and {len(report.group(issues)) - 25} more issue types in the Excel file")

from google.colab import files
files.download(out + ".xlsx")
files.download(out + ".html")

## Notes
- The Excel file has a sheet per category plus a list of affected URLs for every issue. The HTML file is the same report for reading in a browser.
- Spelling and grammar checks can have false positives. Add your brand words in **EXTRA_WORDS**.
- Pages built entirely by JavaScript may look empty to this tool.
- Backlink data needs an export from a tool like Ahrefs, Semrush, Moz or Google Search Console.